In [ ]:
from google.colab import drive
drive.mount('/content/MyDrive')


Drive already mounted at /content/my_drive; to attempt to forcibly remount, call drive.mount("/content/my_drive", force_remount=True).


In [ ]:
# ================================
# MobileNetV2 Image Classification Training (FULL CODE)
# ================================

import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import torch.nn as nn
from torchvision.models import mobilenet_v2
import torch.optim as optim
import os

# ================================
# DEVICE
# ================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ================================
# TRANSFORMS
# ================================
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])
test_transforms = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

# ================================
# DATA LOADING
# ================================
train_dataset = torchvision.datasets.ImageFolder(
    root="/content/drive/MyDrive/processed_data/train",
    transform=train_transforms
)
test_dataset = torchvision.datasets.ImageFolder(
    root="/content/drive/MyDrive/processed_data/test",
    transform=test_transforms
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

print(f"Train size: {len(train_dataset)}, Test size: {len(test_dataset)}")
print("Classes:", train_dataset.classes)

# ================================
# MODEL
# ================================
model = mobilenet_v2(pretrained=True)
num_ftrs = model.classifier[1].in_features
num_classes = len(train_dataset.classes)

model.classifier[1] = nn.Linear(num_ftrs, num_classes)
# চাইলে ড্রপআউট বাড়াতে পারেন
# model.classifier[0] = nn.Dropout(p=0.5)

model = model.to(device)

# ================================
# TRAINING SETTINGS
# ================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-4)

num_epochs = 10

# ================================
# TRAINING LOOP
# ================================
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for inputs, labels in train_loader:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)

    epoch_loss = running_loss / len(train_dataset)
    print(f"\nEpoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")

    # ================================
    # VALIDATION
    # ================================
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (preds == labels).sum().item()

    acc = 100 * correct / total
    print(f"Validation Accuracy: {acc:.2f}%")

# ================================
# SAVE MODEL
# ================================
save_dir = "/content/drive/MyDrive/Model"
os.makedirs(save_dir, exist_ok=True)
save_path = os.path.join(save_dir, "mobilenetv2.pth")
torch.save(model.state_dict(), save_path)
print(f"\n🎉 Model saved successfully at: {save_path}")


Using device: cpu


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=M

Train size: 1600, Test size: 400
Classes: ['HC', 'PD']
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 151MB/s]



Epoch [1/10], Loss: 0.3333
Validation Accuracy: 92.00%

Epoch [2/10], Loss: 0.2273
Validation Accuracy: 95.75%

Epoch [3/10], Loss: 0.1639
Validation Accuracy: 95.75%

Epoch [4/10], Loss: 0.1432
Validation Accuracy: 97.25%

Epoch [5/10], Loss: 0.1399
Validation Accuracy: 96.50%

Epoch [6/10], Loss: 0.1399
Validation Accuracy: 96.25%

Epoch [7/10], Loss: 0.1355
Validation Accuracy: 98.00%

Epoch [8/10], Loss: 0.1013
Validation Accuracy: 97.50%

Epoch [9/10], Loss: 0.0829
Validation Accuracy: 97.50%

Epoch [10/10], Loss: 0.0876
Validation Accuracy: 97.50%

🎉 Model saved successfully at: /content/drive/MyDrive/Model/mobilenetv2.pth


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
